# Clase 3 — Reglas vs. LLM: una decisión de negocio real

En una tienda online llegan cientos de reclamos por chat. Antes de responder, el sistema debe decidir **a qué equipo asignar cada caso, con qué prioridad y en cuánto tiempo atenderlo**.

Una clasificación equivocada se siente en el negocio: un posible fraude puede esperar horas o un ticket normal puede saturar el canal de emergencias. Compararemos sobre los mismos casos:

1. **Reglas:** rápidas, baratas y predecibles.
2. **LLM local:** comprende distintas formas de expresar un problema.
3. **Híbrido:** reglas para límites no negociables + LLM para lenguaje.

Usaremos el mismo **LFM2.5 1.2B Instruct** de la Clase 2.

---
## 1. La política operativa

| Cola | Qué recibe | SLA | Prioridad |
|---|---|---:|---|
| `seguridad` | Compra o acceso que el cliente no reconoce | 15 min | crítica |
| `pagos` | Cobro duplicado, pago rechazado o reintegro pendiente | 2 h | alta |
| `logistica` | Pedido demorado, perdido, dañado o incompleto | 4 h | media |
| `general` | Consultas de producto, cambios y otros casos | 24 h | normal |

Si el mensaje incluye una contraseña, número completo de tarjeta o código de seguridad, debe ir a revisión humana. El sistema no debe procesar secretos.

In [ ]:
import json
import re
import time
import pandas as pd
from IPython.display import display

POLITICA = {
    "seguridad": {"prioridad": "critica", "sla_min": 15},
    "pagos": {"prioridad": "alta", "sla_min": 120},
    "logistica": {"prioridad": "media", "sla_min": 240},
    "general": {"prioridad": "normal", "sla_min": 1440},
}

CASOS = [
    {"id": "T-101", "texto": "Veo una compra en Córdoba y yo estoy en Mendoza.", "esperado": "seguridad"},
    {"id": "T-102", "texto": "Me descontaron dos veces el mismo pedido.", "esperado": "pagos"},
    {"id": "T-103", "texto": "Decía que llegaba el martes; ya es viernes y sigo esperando.", "esperado": "logistica"},
    {"id": "T-104", "texto": "La caja llegó cerrada pero faltaba el cargador.", "esperado": "logistica"},
    {"id": "T-105", "texto": "¿La campera azul también viene en talle M?", "esperado": "general"},
    {"id": "T-106", "texto": "No fui yo quien hizo ese pedido.", "esperado": "seguridad"},
]
display(pd.DataFrame(CASOS))

### Pensá antes de programar

¿Cuáles casos se detectan con una lista de palabras? ¿Cuáles requieren entender la situación? *“Yo estoy en Mendoza* solo es señal de fraude cuando se relaciona con una compra hecha en otra ciudad: importa el significado completo.

---
## 2. Solución A — Reglas de palabras clave

Las reglas son un buen punto de partida para expresiones estables y permiten explicar exactamente cada decisión.

In [ ]:
REGLAS = {
    "seguridad": [
        "no reconozco", 
        "compra desconocida", 
        "me robaron", 
        "fraude"],
    "pagos": [
        "cobro duplicado", 
        "cobraron dos veces", 
        "pago rechazado", 
        "reintegro"],
    "logistica": [
        "no llegó", 
        "demorado", 
        "paquete roto", 
        "pedido incompleto"],
}

def aplicar_politica(cola):
    return {"cola": cola, **POLITICA[cola]}

def clasificar_con_reglas(texto):
    normalizado = texto.lower()
    for cola, expresiones in REGLAS.items():
        for expresion in expresiones:
            if expresion in normalizado:
                return {**aplicar_politica(cola), "metodo": "reglas",
                        "evidencia": f"coincidió: '{expresion}'",
                        "requiere_revision": False}
    return {**aplicar_politica("general"), "metodo": "reglas",
            "evidencia": "ninguna regla coincidió",
            "requiere_revision": False}


display("Mensajes recibidos por el usuario", pd.DataFrame(CASOS))

display("Palabras utilizadas como Reglas", pd.DataFrame(REGLAS))

print("Clasificacion basada en Reglas")
for caso in CASOS:
    r = clasificar_con_reglas(caso["texto"])
    print(caso["id"], "→", r["cola"], "|", r["evidencia"])




La regla no entiende que *“me descontaron dos veces”* equivale a cobro duplicado ni que esperar desde el martes indica demora. Podríamos agregar frases para siempre, pero los clientes encontrarán nuevas formas de decir lo mismo.

---
## 3. Solución B — El LLM local de Clase 2

Usamos exactamente `unsloth/LFM2.5-1.2B-Instruct-GGUF` en Q8. La primera ejecución puede descargar unos 1,25 GB; luego reutiliza la caché. Si faltan dependencias, ejecutá una vez `%pip install -q llama-cpp-python huggingface-hub pandas` y reiniciá el kernel.

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

REPO_ID = "unsloth/LFM2.5-1.2B-Instruct-GGUF"
FILENAME = "LFM2.5-1.2B-Instruct-Q8_0.gguf"

inicio = time.time()
ruta_modelo = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
llm = Llama(model_path=ruta_modelo, n_ctx=4096, n_gpu_layers=0, verbose=False)
print(f"LFM2.5 listo en {time.time() - inicio:.1f} s")
print("Modelo local:", ruta_modelo)

### Un contrato pequeño y verificable

El LLM no decide libremente el SLA. Solo interpreta el mensaje y propone una de cuatro colas. Python valida la salida y aplica la política:

```text
mensaje → LLM interpreta → Python valida → política asigna prioridad y SLA
```

Así el modelo no puede inventar equipos ni tiempos de atención.

In [ ]:
COLAS_VALIDAS = set(POLITICA)

PROMPT_CLASIFICADOR = """
Sos el clasificador de consultas/triage de una tienda online argentina.
Elegí una cola por el significado del mensaje:
- seguridad: compra, pedido o acceso que la persona no realizó o reconoce.
- pagos: cobro duplicado, pago rechazado o devolución pendiente.
- logistica: entrega demorada, perdida, dañada o incompleta.
- general: producto, cambios u otros casos.

Como responder:
cola: decide 1 opcion
motivo: explicar decision en 30 tokens
Respondé ÚNICAMENTE JSON válido, sin Markdown:
{"cola":"opcion","motivo":"explicacion"}
""".strip()

def extraer_json(texto):
    inicio, fin = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fin == -1:
        raise ValueError("la respuesta no contiene JSON")
    return json.loads(texto[inicio:fin + 1])

def clasificar_con_llm(texto):
    inicio = time.time()
    salida = llm.create_chat_completion(
        messages=[{"role": "system", "content": PROMPT_CLASIFICADOR},
                  {"role": "user", "content": texto}],
        temperature=0, max_tokens=80)
    salida_cruda = salida["choices"][0]["message"]["content"].strip()
    try:
        datos = extraer_json(salida_cruda)
        cola = datos.get("cola")
        if cola not in COLAS_VALIDAS:
            raise ValueError(f"cola no permitida: {cola}")
    except (json.JSONDecodeError, ValueError) as error:
        return {**aplicar_politica("general"), "metodo": "llm_local",
                "evidencia": f"salida inválida: {error}",
                "requiere_revision": True, "salida_cruda": salida_cruda,
                "duracion_s": round(time.time() - inicio, 2)}
    return {**aplicar_politica(cola), "metodo": "llm_local",
            "evidencia": datos.get("motivo", "sin motivo"),
            "requiere_revision": False, "salida_cruda": salida_cruda,
            "duracion_s": round(time.time() - inicio, 2)}

In [ ]:
display(pd.DataFrame(CASOS))

In [ ]:
clasificar_con_llm("La caja llegó cerrada pero faltaba el cargador.")

`duracion_s` y `salida_cruda` hacen visible que LFM2.5 se ejecutó realmente. Si el JSON falla, el sistema no oculta el problema: exige revisión.

---
## 4. Comparación lado a lado

Ambos reciben los mismos mensajes. Mediremos aciertos y un dato operativo: cuántos casos críticos quedarían esperando en la cola general.

In [ ]:
filas = []
for caso in CASOS:
    reglas = clasificar_con_reglas(caso["texto"])
    modelo = clasificar_con_llm(caso["texto"])
    filas.append({"id": caso["id"], "mensaje": caso["texto"],
        "esperado": caso["esperado"], "reglas": reglas["cola"],
        "reglas_ok": reglas["cola"] == caso["esperado"],
        "llm": modelo["cola"], "llm_ok": modelo["cola"] == caso["esperado"],
        "motivo_llm": modelo["evidencia"], "segundos_llm": modelo["duracion_s"]})

comparacion = pd.DataFrame(filas)
display(comparacion)
print(f"Exactitud reglas: {comparacion['reglas_ok'].mean():.0%}")
print(f"Exactitud LLM:    {comparacion['llm_ok'].mean():.0%}")
perdidos = comparacion.query("esperado == 'seguridad' and reglas != 'seguridad'")
print("Casos críticos perdidos por reglas:", len(perdidos))

### Qué se vuelve palpable

- Las **reglas** responden casi instantáneamente, pero solo reconocen lo anticipado literalmente.
- El **LLM** tarda más, pero relaciona conceptos: dos débitos son un duplicado; esperar días es demora.
- El LLM propone la cola; Python conserva la autoridad sobre la política.
- Local significa que el texto no viaja a una API externa, no que podamos ignorar la privacidad.

> La pregunta útil no es “¿qué tecnología es mejor?”, sino “¿qué error puede tolerar esta decisión?”.

---
## 5. Solución recomendada — Híbrida

Las reglas son ideales para controles que no admiten interpretación. El LLM es útil para comprender el motivo del reclamo.

In [ ]:
PATRONES_SENSIBLES = {
    "posible tarjeta completa": r"(?<!\d)(?:\d[ -]?){15,16}(?!\d)",
    "contraseña declarada": r"(?:mi )?(?:clave|contraseña)\s*(?:es|:)",
    "código de seguridad": r"(?:cvv|cvc|código de seguridad)\s*(?:es|:)?\s*\d{3,4}",
}

def detectar_dato_sensible(texto):
    for nombre, patron in PATRONES_SENSIBLES.items():
        if re.search(patron, texto, flags=re.IGNORECASE):
            return nombre
    return None

def clasificar_hibrido(texto):
    sensible = detectar_dato_sensible(texto)
    if sensible:
        return {**aplicar_politica("seguridad"), "metodo": "regla_de_seguridad",
                "evidencia": sensible, "requiere_revision": True}
    return clasificar_con_llm(texto)

pruebas = ["Me descontaron dos veces el pedido.",
           "Mi contraseña es mate2026, pero no puedo entrar.",
           "La tarjeta 4509 9535 6623 3704 fue rechazada."]
for texto in pruebas:
    print("MENSAJE:", texto)
    print("DECISIÓN:", clasificar_hibrido(texto), "\n")

En los mensajes sensibles, el LLM **ni siquiera se ejecuta**: una regla corta el flujo y exige revisión. En el reclamo natural, el LLM aporta comprensión. Esa división es una decisión de arquitectura, no un truco de prompting.

---
## 6. Laboratorio — Sentí dónde se rompe cada enfoque

Escribí un reclamo como un cliente real, sin copiar palabras de la política. Compará la salida y respondé: ¿la regla encontró algo literal?, ¿qué significado infirió el LLM?, ¿qué pasaría si se equivoca?, ¿agregarías una regla, usarías LLM o pedirías revisión?

In [ ]:
mi_reclamo = "El paquete figura entregado, pero en casa no lo recibió nadie."
display(pd.DataFrame([
    {"enfoque": "reglas", **clasificar_con_reglas(mi_reclamo)},
    {"enfoque": "híbrido", **clasificar_hibrido(mi_reclamo)},
]))

### Desafío breve

Creá tres mensajes: uno que las reglas resuelvan bien, uno sin palabras clave que requiera contexto y uno con un dato sensible. Ejecutalos con `clasificar_hibrido`. Justificá el camino considerando exactitud, latencia, mantenimiento y daño de un error.

---
## 7. Ejercicio integrador — Triage de mensajes en un centro médico

Un centro médico recibe mensajes por WhatsApp antes de que intervenga el personal administrativo. El sistema **no diagnostica ni recomienda tratamientos**: solo organiza la bandeja de entrada.

La decisión de negocio es asignar cada mensaje a una cola:

| Cola | Ejemplos | Acción operativa |
|---|---|---|
| `urgente` | dificultad para respirar, pérdida de conciencia, dolor intenso repentino | detener automatización y alertar a una persona |
| `turnos` | pedir, cambiar o cancelar una cita | enviar a agenda |
| `resultados` | consultar si un estudio o análisis está disponible | enviar a recepción de estudios |
| `administracion` | cobertura, autorización, factura o documentación | enviar a administración |
| `otro` | mensaje insuficiente o fuera de las categorías | revisión humana |

_
> **Importante:** En este dominio, un falso negativo urgente es mucho más costoso que una derivación innecesaria. *Ante duda, el sistema debe escalar* y nunca afirmar que una persona está o no está en una emergencia.

### Parte A — Escribí las reglas

Completá patrones para señales explícitas que no querés dejar únicamente a una interpretación probabilística. Evitá palabras demasiado generales como `dolor`, que producirían muchas falsas alarmas.

Después probá las reglas con los mensajes de ejemplo y observá qué expresiones indirectas no detectan.

In [ ]:
REGLAS_MEDICAS = {
    "urgente": [
        # TODO: agregá al menos 3 expresiones explícitas de alarma
        # Ejemplo: "no puede respirar"
    ],
    "turnos": [
        # TODO: agregá al menos 2 expresiones
    ],
    "resultados": [
        # TODO: agregá al menos 2 expresiones
    ],
    "administracion": [
        # TODO: agregá al menos 2 expresiones
    ],
}

def clasificar_reglas_medicas(texto):
    normalizado = texto.lower()
    for cola, expresiones in REGLAS_MEDICAS.items():
        for expresion in expresiones:
            if expresion.lower() in normalizado:
                return {"cola": cola, "evidencia": expresion}
    return {"cola": "otro", "evidencia": "sin coincidencia"}

CASOS_MEDICOS = [
    "Quiero cambiar el turno del jueves.",
    "¿Ya está el resultado de mi análisis?",
    "Mi obra social pide una autorización.",
    "Mi papá se desmayó y todavía no reacciona.",
    "Desde hace unos minutos siente que el aire no le alcanza.",
]

for mensaje in CASOS_MEDICOS:
    print(mensaje, "→", clasificar_reglas_medicas(mensaje))

### Parte B — Diseñá el prompt para LFM2.5

Completá el contrato del clasificador. El prompt debe definir el rol, las categorías, el criterio conservador para urgencias, los límites clínicos y un JSON fácil de validar. No incluyas diagnósticos ni instrucciones de tratamiento.

In [ ]:
PROMPT_MEDICO = """
ROL:
Sos ...

OBJETIVO:
Clasificá ...

CATEGORÍAS:
- urgente: ...
- turnos: ...
- resultados: ...
- administracion: ...
- otro: ...

CRITERIO DE SEGURIDAD:
Si ...

LÍMITES:
- No diagnostiques.
- No recomiendes medicamentos ni tratamientos.
- No inventes información.
- ...

FORMATO:
"cola": decide 1 opcion en CATEGORIAS
"motivo": explica decicion con máximo 12 palabras
Respondé únicamente JSON válido:
{"cola":"opcion","motivo":"explicacion"}
""".strip()

# Reutilizamos el objeto `llm` cargado anteriormente.
def probar_prompt_medico(mensaje):
    salida = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": PROMPT_MEDICO},
            {"role": "user", "content": mensaje},
        ],
        temperature=0,
        max_tokens=80,
    )
    return salida["choices"][0]["message"]["content"].strip()

for mensaje in CASOS_MEDICOS:
    print("MENSAJE:", mensaje)
    print("LLM:", probar_prompt_medico(mensaje), "\n")

### Parte C — Defendé tu arquitectura

Compará las reglas y el LLM en los cinco mensajes. Luego respondé:

1. ¿Qué señales deben permanecer como reglas aunque el LLM las reconozca?
2. ¿Qué expresiones indirectas comprendió mejor el modelo?
3. ¿Qué salida usarías si el JSON es inválido o la categoría es `otro`?
4. ¿Qué datos personales evitarías guardar en la traza?
5. Dibujá el flujo híbrido final e indicá exactamente dónde interviene una persona.

**Criterio de logro:** la propuesta es exitosa si mantiene la clasificación acotada, escala conservadoramente los casos de posible urgencia y deja claro que el sistema organiza mensajes, pero no realiza evaluación médica.

---
## Cierre

| Usá… | Cuando… | Ejemplo |
|---|---|---|
| **Reglas** | el patrón es estable y no negociable | bloquear secretos |
| **LLM local** | importa el significado y hay muchas formas de expresarlo | motivo del reclamo |
| **Híbrido** | necesitás comprensión con límites deterministas | regla → LLM → validación → SLA |
| **Humano** | la salida es inválida, sensible o el costo del error es alto | revisar credenciales |

La solución útil no es “poner un LLM”. Es diseñar qué parte interpreta, qué parte conserva el control y qué ocurre cuando algo falla.